# 総研大 2024年度 講義「シミュレーション天文学」 / SOKEN-dai FY2024 Lecture Course: “Simulation Astronomy”
### 担当：馬場淳一 (国立天文台 科学研究部/鹿児島大学 天の川銀河研究センター) / Junichi Baba (D. of Science, NAOJ / Kagoshima University)

This code is provided for educational purposes only. Users are encouraged to utilize it at their own risk. I take no responsibility for any results obtained from research or analyses conducted using this code. It is essential to deepen your understanding and verify the results independently. Additionally, it is recommended to cross-reference with other reliable resources to ensure the appropriate methodologies are applied.

このコードは教育目的で提供されています。使用にあたっては、自己責任で行ってください。提供されたコードを基に行った研究や解析によって得られた結果について、私は一切の責任を負いません。使用する際には、必ず自身の理解を深め、結果を検証してください。また、他の信頼できるリソースと照らし合わせて、適切な方法論を選択することをお勧めします。

## 学習目標と計算上の前提

この実装はAB予測＋AM修正1回のPECE法で、陰的AM方程式を収束まで解く方式ではありません。完全な陰的法の安定性をそのまま適用しないでください。4次法は初期値に加え3ステップをRK4で準備します。

検証課題と判定基準は [演習ガイド](../docs/exercises.md) を参照してください。


このNotebookでは線形自律方程式 $y'=-\alpha y$ を比較します。この問題ではHeun法と陽的中点法の更新多項式が同じなので、図の曲線は丸め誤差を除き重なります。一般の非線形問題で同じ方法になるわけではありません。


In [ ]:
import numpy as np
import time 
import matplotlib.pyplot as plt
import matplotlib.cm as cm
parameters = {
    'axes.labelsize': 26,           # Set the font size of the x-axis and y-axis labels to 26
    'xtick.labelsize': 24,          # Set the font size of the x-axis tick labels to 24
    'ytick.labelsize': 24,          # Set the font size of the y-axis tick labels to 24
    'axes.titlesize': 24,           # Set the font size of the axis titles (not the overall graph title) to 24
    'figure.titlesize': 24,         # Set the font size of the overall graph title to 24
    'legend.fontsize': 22,          # Set the font size of the legend to 22
    'figure.figsize': [10, 6],      # Set the default figure size (width 10 inches, height 6 inches)
    'axes.linewidth': 2.0           # Set the linewidth of the axes
}
plt.rcParams.update(parameters)


In [ ]:
def f(t, y, alpha):
    """
    Defines the right-hand side of the differential equation dy/dt = -alpha * y.

    Parameters:
        t (float): The current time
        y (float): The current value of y
        alpha (float): The coefficient alpha in the equation

    Returns:
        float: The value of dy/dt
    """
    return -alpha * y

def analytical_solution(t, alpha):
    """
    Computes the analytical solution y(t) = e^(-alpha * t).

    Parameters:
        t (float or np.ndarray): The time at which to evaluate the solution
        alpha (float): The coefficient alpha in the equation

    Returns:
        float or np.ndarray: The analytical solution y(t)
    """
    return np.exp(-alpha * t)


## 前進Euler法の復習

更新式と説明は [ex01](ex01%20euler_accuracy.ipynb) を参照してください。以下では比較のため同じ実装を定義します。


In [ ]:
# Define the Forward Euler Method Function
def euler_method(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using the explicit Forward Euler method.

    Parameters:
        f (callable): Function representing dy/dt = f(t, y)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    for n in range(N):
        t_n = t_values[n]
        y_n = y_values[n]
        y_new = y_n + h * f(t_n, y_n, alpha)
        y_values[n + 1] = y_new

    return t_values, y_values


## Heun法の復習

式と説明は [ex02](ex02%20rk2_accuracy.ipynb) を参照してください。以下では比較用にHeun法を定義します。


In [ ]:
def rk2_heun_method(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using Heun's method (explicit midpoint method).

    Parameters:
        f (callable): Function representing dy/dt = f(t, y, alpha)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    for n in range(N):
        t_n = t_values[n]
        y_n = y_values[n]

        # Predictor step (Forward Euler)
        y_predictor = y_n + h * f(t_n, y_n, alpha)

        # Corrector step (average slope)
        t_new = t_n + h
        y_corrector = y_n + (h / 2) * (f(t_n, y_n, alpha) + f(t_new, y_predictor, alpha))

        y_values[n + 1] = y_corrector

    return t_values, y_values


## 陽的中点法の復習

式とHeun法との違いは [ex02](ex02%20rk2_accuracy.ipynb) を参照してください。以下では比較用に陽的中点法を定義します。


In [ ]:
def rk2_midpoint_method(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using the RK2 method (midpoint method).

    Parameters:
        f (callable): Function representing dy/dt = f(t, y, alpha)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    for n in range(N):
        t_n = t_values[n]
        y_n = y_values[n]

        # Predictor step (Euler's step to midpoint)
        k1 = f(t_n, y_n, alpha)
        y_mid = y_n + (h / 2) * k1  # Midpoint value

        # Corrector step (use slope at the midpoint)
        k2 = f(t_n + h / 2, y_mid, alpha)
        y_new = y_n + h * k2  # Update using the slope at midpoint

        y_values[n + 1] = y_new

    return t_values, y_values


## 古典RK4の復習

係数と更新式は [ex03](ex03%20rk4_accuracy.ipynb) を参照してください。以下のRK4は、比較とAdams法の開始値の計算に使います。


In [ ]:
def rk4_method(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using the 4th-order Runge-Kutta method (RK4).

    Parameters:
        f (callable): Function representing dy/dt = f(t, y, alpha)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    for n in range(N):
        t_n = t_values[n]
        y_n = y_values[n]

        # Calculate the RK4 coefficients
        k1 = f(t_n, y_n, alpha)
        k2 = f(t_n + h / 2, y_n + h / 2 * k1, alpha)
        k3 = f(t_n + h / 2, y_n + h / 2 * k2, alpha)
        k4 = f(t_n + h, y_n + h * k3, alpha)

        # Update the solution for the next time step
        y_new = y_n + (h / 6) * (k1 + 2 * k2 + 2 * k3 + k4)
        y_values[n + 1] = y_new

    return t_values, y_values


## 2次Adams予測修正法（AB2–AM2、修正1回）

$f_n=f(t_n,y_n)$ と書くと、予測と修正は
$$y_{n+1}^{(p)}=y_n+\frac h2(3f_n-f_{n-1}),$$
$$y_{n+1}=y_n+\frac h2\{f(t_{n+1},y_{n+1}^{(p)})+f_n\}.$$

予測値で右辺を1回評価するPECE型であり、陰的AM2方程式を収束するまで解く方法ではありません。大域誤差は通常2次です。Heun法・陽的中点法も2次であり、同じ次数の手法間の誤差の大小は問題に依存します。

線形試験方程式 $y'=\lambda y$ の負の実軸上で、安定区間は $-2\le h\lambda\le0$ です。これは前進Euler法や2段2次RK法と同じ区間で、安定性が一律に向上するわけではありません。端点では減衰しない成分があり、減衰の実演では区間の内部を用います。


### 実装と関数評価回数

RK4で1ステップ進めて2つの開始値を用意します。以後、各ステップで $f_n,f_{n-1},f(t_{n+1},y_{n+1}^{(p)})$ を計算するため関数評価は3回です。過去の確定した傾きを保存する実装なら、新しい予測点と修正点の評価を合わせて定常的に2回にできます。本教材の時間比較は、履歴を毎回再計算する実装の結果です。


In [ ]:
def adams_bashforth_moulton_2nd(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using the 2nd-order Adams-Bashforth-Moulton predictor-corrector method.

    Parameters:
        f (callable): Function representing dy/dt = f(t, y, alpha)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    # Start with RK4 to get the second point (since Adams methods require two initial points)
    t_values_rk4, y_values_rk4 = rk4_method(f, y0, t0, t0 + h, h, alpha)
    y_values[1] = y_values_rk4[-1]

    for n in range(1, N):
        t_n = t_values[n]
        t_np1 = t_values[n + 1]  # t_n+1
        y_n = y_values[n]
        y_nm1 = y_values[n - 1]

        # Predictor step (Adams-Bashforth 2nd order)
        f_n = f(t_n, y_n, alpha)
        f_nm1 = f(t_n - h, y_nm1, alpha)
        y_predictor = y_n + (h / 2) * (3 * f_n - f_nm1)

        # Corrector step (Adams-Moulton 2nd order)
        f_predictor = f(t_np1, y_predictor, alpha)
        y_corrector = y_n + (h / 2) * (f_predictor + f_n)

        # Update the solution
        y_values[n + 1] = y_corrector

    return t_values, y_values


In [ ]:
# Main function to execute the comparison and analysis
def main():
    # Initial conditions and settings
    y0 = 1.0
    t0 = 0.0
    t_end = 1.0
    step_sizes = [0.1, 0.05, 0.025, 0.0125, 0.00625, 0.003125, 0.0015625, 
                  0.00078125, 0.000390625, 0.0001953125, 0.0001]
    alphas = [2]  # Different alpha values for comparison
    
    # Prepare for plotting (two subplots: left for error, right for time)
    size = 10
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(2*size, size))
    
    # Define colors for different alpha values
    colors = cm.viridis(np.linspace(0, 1, len(alphas)))
    
    # Iterate over different alpha values
    for idx, alpha in enumerate(alphas):
        # Calculate the analytical solution at t_end
        y_true = analytical_solution(t_end, alpha)
        print(f"Analytical solution y({t_end}) with alpha={alpha} = {y_true:.10f}\n")
        
        # Lists to store errors and computation times for each method and step size
        euler_errors = []
        heun_errors = []
        midpoint_errors = []
        rk4_errors = []
        adams_errors = []
        euler_times = []
        heun_times = []
        midpoint_times = []
        rk4_times = []
        adams_times = []
        
        # Print header
        print(f"Comparison of Methods for alpha = {alpha}:")
        print(f"{'Step Size h':>15} | {'Euler Error':>15} | {'Heun Error':>15} | {'Midpoint Error':>15} | {'RK4 Error':>15} | {'Adams Error':>15}")
        print("-" * 105)
        
        # Apply each method for every step size
        for h in step_sizes:
            # Euler Method
            start_time = time.perf_counter()
            _, y_values_euler = euler_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            euler_time = end_time - start_time
            euler_times.append(euler_time)
            y_euler = y_values_euler[-1]
            euler_error = abs(y_euler - y_true) / abs(y_true)
            euler_errors.append(euler_error)

            # Heun Method
            start_time = time.perf_counter()
            _, y_values_heun = rk2_heun_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            heun_time = end_time - start_time
            heun_times.append(heun_time)
            y_heun = y_values_heun[-1]
            heun_error = abs(y_heun - y_true) / abs(y_true)
            heun_errors.append(heun_error)

            # Midpoint Method
            start_time = time.perf_counter()
            _, y_values_midpoint = rk2_midpoint_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            midpoint_time = end_time - start_time
            midpoint_times.append(midpoint_time)
            y_midpoint = y_values_midpoint[-1]
            midpoint_error = abs(y_midpoint - y_true) / abs(y_true)
            midpoint_errors.append(midpoint_error)

            # RK4 Method
            start_time = time.perf_counter()
            _, y_values_rk4 = rk4_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            rk4_time = end_time - start_time
            rk4_times.append(rk4_time)
            y_rk4 = y_values_rk4[-1]
            rk4_error = abs(y_rk4 - y_true) / abs(y_true)
            rk4_errors.append(rk4_error)

            # Adams-Bashforth-Moulton Method
            start_time = time.perf_counter()
            _, y_values_adams = adams_bashforth_moulton_2nd(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            adams_time = end_time - start_time
            adams_times.append(adams_time)
            y_adams = y_values_adams[-1]
            adams_error = abs(y_adams - y_true) / abs(y_true)
            adams_errors.append(adams_error)

            # Print results for each step size
            print(f"{h:15.5e} | {euler_error:15.10e} | {heun_error:15.10e} | {midpoint_error:15.10e} | {rk4_error:15.10e} | {adams_error:15.10e}")
        
        print("\n")  # Add space between different alpha values
        
        # Plot relative errors for each method on a log-log scale (left plot)
        ax1.loglog(step_sizes, euler_errors, 'o-', markersize=10, lw=4, color='#000000', label=f'Euler alpha={alpha}')
        ax1.loglog(step_sizes, heun_errors, 's--', markersize=10, lw=4, color='#E69F00', label=f'Heun alpha={alpha}')
        ax1.loglog(step_sizes, midpoint_errors, '^:', markersize=10, lw=4, color='#56B4E9', label=f'Midpoint alpha={alpha}')
        ax1.loglog(step_sizes, rk4_errors, 'D-.', markersize=10, lw=4, color='#009E73', label=f'RK4 alpha={alpha}')
        ax1.loglog(step_sizes, adams_errors, 'v--', markersize=10, lw=4, color='#0072B2', label=f'Adams alpha={alpha}')
        
        # Plot computation times for each method on a log-log scale (right plot)
        ax2.loglog(step_sizes, euler_times, 'o-', markersize=10, lw=4, color='#000000', label=f'Euler alpha={alpha}')
        ax2.loglog(step_sizes, heun_times, 's--', markersize=10, lw=4, color='#E69F00', label=f'Heun alpha={alpha}')
        ax2.loglog(step_sizes, midpoint_times, '^:', markersize=10, lw=4, color='#56B4E9', label=f'Midpoint alpha={alpha}')
        ax2.loglog(step_sizes, rk4_times, 'D-.', markersize=10, lw=4, color='#009E73', label=f'RK4 alpha={alpha}')
        ax2.loglog(step_sizes, adams_times, 'v--', markersize=10, lw=4, color='#0072B2', label=f'Adams alpha={alpha}')
    
    # Set labels and titles for the left plot (relative error)
    ax1.set_xlabel('Step Size h')
    ax1.set_ylabel('Relative Error')
    ax1.set_title('Relative Error vs. Step size')
    ax1.grid(True, which="both", ls="--")
    
    # Set labels and titles for the right plot (computation time)
    ax2.set_xlabel('Step Size h')
    ax2.set_ylabel('Computation Time (s)')
    ax2.set_title('Computation Time vs. Step size')
    ax2.grid(True, which="both", ls="--")
    
    # Add legends
    ax1.legend()
    ax2.legend()
    
    # Display the plots
    plt.tight_layout()
    plt.show()
if __name__ == "__main__":
    main()


## 4次Adams予測修正法（AB4–AM4、修正1回）

$$y_{n+1}^{(p)}=y_n+\frac h{24}(55f_n-59f_{n-1}+37f_{n-2}-9f_{n-3}),$$
$$y_{n+1}=y_n+\frac h{24}\{9f(t_{n+1},y_{n+1}^{(p)})+19f_n-5f_{n-1}+f_{n-2}\}.$$

通常は4次の大域誤差を持ちます。修正は1回であり、完全に解いた陰的AM4の安定性とは異なります。線形試験方程式の負の実軸上の安定区間は約 $[-1.285,0]$ で、古典RK4の約 $[-2.785,0]$ より狭い区間です。精度の次数、安定性、計算時間は別々に比較してください。


### 開始値と計算コスト

初期値に加えてRK4の3ステップで計4点を用意します。総ステップ数が3以下の場合は、必要な開始ステップだけを実行します。

この実装は過去4点の傾きと予測点の傾きを再評価するため、開始計算を除いて1ステップ5回の関数評価を行います。RK4の4回より多く、必ず高速になるわけではありません。傾きの履歴を保存すれば、予測点と修正点での評価の計2回に削減できます。


In [ ]:
def adams_bashforth_moulton_4th(f, y0, t0, t_end, h, alpha):
    """
    Solve the ODE dy/dt = f(t, y) using the 4th-order Adams-Bashforth-Moulton predictor-corrector method.

    Parameters:
        f (callable): Function representing dy/dt = f(t, y, alpha)
        y0 (float): Initial condition y(t0) = y0
        t0 (float): Initial time
        t_end (float): End time
        h (float): Step size
        alpha (float): Parameter in the ODE

    Returns:
        t_values (numpy.ndarray): Array of time points
        y_values (numpy.ndarray): Array of numerical solutions at each time point
    """
    if not (np.isfinite(h) and h > 0 and np.isfinite(t0) and np.isfinite(t_end) and t_end > t0):
        raise ValueError("Require finite t_end > t0 and h > 0")
    N = max(1, int(round((t_end - t0) / h)))
    h = (t_end - t0) / N  # Actual uniform step matches returned times
    t_values = np.linspace(t0, t_end, N + 1)
    y_values = np.zeros(N + 1)
    y_values[0] = y0

    # Use RK4 to take three startup steps (four points including the initial value)
    for i in range(min(3, N)):
        t_values_rk4, y_values_rk4 = rk4_method(f, y_values[i], t0 + i * h, t0 + (i + 1) * h, h, alpha)
        y_values[i + 1] = y_values_rk4[-1]

    for n in range(3, N):
        t_n = t_values[n]
        t_np1 = t_values[n + 1]  # t_n+1
        y_n = y_values[n]
        y_nm1 = y_values[n - 1]
        y_nm2 = y_values[n - 2]
        y_nm3 = y_values[n - 3]

        # Predictor step (Adams-Bashforth 4th order)
        f_n = f(t_n, y_n, alpha)
        f_nm1 = f(t_n - h, y_nm1, alpha)
        f_nm2 = f(t_n - 2 * h, y_nm2, alpha)
        f_nm3 = f(t_n - 3 * h, y_nm3, alpha)
        y_predictor = y_n + (h / 24) * (55 * f_n - 59 * f_nm1 + 37 * f_nm2 - 9 * f_nm3)

        # Corrector step (Adams-Moulton 4th order)
        f_predictor = f(t_np1, y_predictor, alpha)
        y_corrector = y_n + (h / 24) * (9 * f_predictor + 19 * f_n - 5 * f_nm1 + f_nm2)

        # Update the solution
        y_values[n + 1] = y_corrector

    return t_values, y_values


In [ ]:
# Main function to execute the comparison and analysis
def main():
    # Initial conditions and settings
    y0 = 1.0
    t0 = 0.0
    t_end = 1.0
    step_sizes = [0.1, 0.05, 0.025, 0.0125, 0.00625, 0.003125, 0.0015625, 
                  0.00078125, 0.000390625, 0.0001953125, 0.0001]
    alphas = [2]  # Different alpha values for comparison
    
    # Prepare for plotting (two subplots: left for error, right for time)
    size = 10
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(2*size, size))
    
    # Define colors for different alpha values
    colors = cm.viridis(np.linspace(0, 1, len(alphas)))
    
    # Iterate over different alpha values
    for idx, alpha in enumerate(alphas):
        # Calculate the analytical solution at t_end
        y_true = analytical_solution(t_end, alpha)
        print(f"Analytical solution y({t_end}) with alpha={alpha} = {y_true:.10f}\n")
        
        # Lists to store errors and computation times for each method and step size
        euler_errors = []
        heun_errors = []
        midpoint_errors = []
        rk4_errors = []
        adams2_errors = []
        adams4_errors = []
        euler_times = []
        heun_times = []
        midpoint_times = []
        rk4_times = []
        adams2_times = []
        adams4_times = []
        
        # Print header
        print(f"Comparison of Methods for alpha = {alpha}:")
        print(f"{'Step Size h':>15} | {'Euler Error':>15} | {'Heun Error':>15} | {'Midpoint Error':>15} | {'RK4 Error':>15} | {'Adams2 Error':>15} | {'Adams4 Error':>15}")
        print("-" * 120)
        
        # Apply each method for every step size
        for h in step_sizes:
            # Euler Method
            start_time = time.perf_counter()
            _, y_values_euler = euler_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            euler_time = end_time - start_time
            euler_times.append(euler_time)
            y_euler = y_values_euler[-1]
            euler_error = abs(y_euler - y_true) / abs(y_true)
            euler_errors.append(euler_error)

            # Heun Method
            start_time = time.perf_counter()
            _, y_values_heun = rk2_heun_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            heun_time = end_time - start_time
            heun_times.append(heun_time)
            y_heun = y_values_heun[-1]
            heun_error = abs(y_heun - y_true) / abs(y_true)
            heun_errors.append(heun_error)

            # Midpoint Method
            start_time = time.perf_counter()
            _, y_values_midpoint = rk2_midpoint_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            midpoint_time = end_time - start_time
            midpoint_times.append(midpoint_time)
            y_midpoint = y_values_midpoint[-1]
            midpoint_error = abs(y_midpoint - y_true) / abs(y_true)
            midpoint_errors.append(midpoint_error)

            # RK4 Method
            start_time = time.perf_counter()
            _, y_values_rk4 = rk4_method(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            rk4_time = end_time - start_time
            rk4_times.append(rk4_time)
            y_rk4 = y_values_rk4[-1]
            rk4_error = abs(y_rk4 - y_true) / abs(y_true)
            rk4_errors.append(rk4_error)

            # Adams-Bashforth-Moulton 2nd Order Method
            start_time = time.perf_counter()
            _, y_values_adams2 = adams_bashforth_moulton_2nd(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            adams2_time = end_time - start_time
            adams2_times.append(adams2_time)
            y_adams2 = y_values_adams2[-1]
            adams2_error = abs(y_adams2 - y_true) / abs(y_true)
            adams2_errors.append(adams2_error)

            # Adams-Bashforth-Moulton 4th Order Method
            start_time = time.perf_counter()
            _, y_values_adams4 = adams_bashforth_moulton_4th(f, y0, t0, t_end, h, alpha)
            end_time = time.perf_counter()
            adams4_time = end_time - start_time
            adams4_times.append(adams4_time)
            y_adams4 = y_values_adams4[-1]
            adams4_error = abs(y_adams4 - y_true) / abs(y_true)
            adams4_errors.append(adams4_error)

            # Print results for each step size
            print(f"{h:15.5e} | {euler_error:15.10e} | {heun_error:15.10e} | {midpoint_error:15.10e} | {rk4_error:15.10e} | {adams2_error:15.10e} | {adams4_error:15.10e}")
        
        print("\n")  # Add space between different alpha values
        
        # Plot relative errors for each method on a log-log scale (left plot)
        ax1.loglog(step_sizes, euler_errors, 'o-', markersize=10, lw=4, color='#000000', label=f'Euler alpha={alpha}')
        ax1.loglog(step_sizes, heun_errors, 's--', markersize=10, lw=4, color='#E69F00', label=f'Heun alpha={alpha}')
        ax1.loglog(step_sizes, midpoint_errors, '^:', markersize=10, lw=4, color='#56B4E9', label=f'Midpoint alpha={alpha}')
        ax1.loglog(step_sizes, rk4_errors, 'D-.', markersize=10, lw=4, color='#009E73', label=f'RK4 alpha={alpha}')
        ax1.loglog(step_sizes, adams2_errors, 'v--', markersize=10, lw=4, color='#0072B2', label=f'Adams2 alpha={alpha}')
        ax1.loglog(step_sizes, adams4_errors, 'P:', markersize=10, lw=4, color='#D55E00', label=f'Adams4 alpha={alpha}')
        
        # Plot computation times for each method on a log-log scale (right plot)
        ax2.loglog(step_sizes, euler_times, 'o-', markersize=10, lw=4, color='#000000', label=f'Euler alpha={alpha}')
        ax2.loglog(step_sizes, heun_times, 's--', markersize=10, lw=4, color='#E69F00', label=f'Heun alpha={alpha}')
        ax2.loglog(step_sizes, midpoint_times, '^:', markersize=10, lw=4, color='#56B4E9', label=f'Midpoint alpha={alpha}')
        ax2.loglog(step_sizes, rk4_times, 'D-.', markersize=10, lw=4, color='#009E73', label=f'RK4 alpha={alpha}')
        ax2.loglog(step_sizes, adams2_times, 'v--', markersize=10, lw=4, color='#0072B2', label=f'Adams2 alpha={alpha}')
        ax2.loglog(step_sizes, adams4_times, 'P:', markersize=10, lw=4, color='#D55E00', label=f'Adams4 alpha={alpha}')
    
    # Set labels and titles for the left plot (relative error)
    ax1.set_xlabel('Step Size h')
    ax1.set_ylabel('Relative Error')
    ax1.set_title('Relative Error vs. Step size')
    ax1.grid(True, which="both", ls="--")
    
    # Set labels and titles for the right plot (computation time)
    ax2.set_xlabel('Step Size h')
    ax2.set_ylabel('Computation Time (s)')
    ax2.set_title('Computation Time vs. Step size')
    ax2.grid(True, which="both", ls="--")
    
    # Add legends
    ax1.legend()
    ax2.legend()
    
    # Display the plots
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()
